# AR(p) processes and forward sliding-window embeddings

This notebook simulates a stationary AR(p) process

$$X_t = c + \phi_1X_{t-1}+\cdots+\phi_pX_{t-p}+\varepsilon_t,$$

using bounded uniform innovations. It plots the time series and the forward embeddings

$$Y_t^{(k)}=(X_t,X_{t+1},\ldots,X_{t+k-1})$$

for $k=2$ and $k=3$. The 3D Plotly figure can be rotated, zoomed, and panned.

Geometric overlays:

- **Middle hyperplane:** the zero-innovation AR relation.
- **Support boundaries:** the two border hyperplanes obtained by adding the lower and upper uniform-innovation bounds.

> An exact AR innovation hyperplane can be drawn only when the visible window contains all required lags, i.e. $p\leq k-1$. Thus $k=2$ supports an exact AR(1) line, while $k=3$ supports AR(1) and AR(2) planes. The notebook reports when a chosen order is too large for an exact overlay.

In [1]:
# If needed, uncomment the next line once:
# %pip install numpy matplotlib plotly ipywidgets

import ast

import numpy as np
import ipywidgets as widgets

from IPython.display import display, clear_output
from Code.Code_AR_processes.functions import (
    simulate_uniform_ar,
    sliding_window_point_cloud,
    companion_matrix,
    is_stationary,
)

from Code.Code_AR_processes.plotting_functions import (
    plot_time_series,
    plot_embedding_k2,
    plot_embedding_k3,
)

ModuleNotFoundError: No module named 'Code'

## Simulation and geometry functions

In [ ]:
def parse_phi(text):
    """Parse entries such as '0.7', '0.6, -0.2', or '[0.6, -0.2]' safely."""
    text = text.strip()
    if not text:
        raise ValueError("Please enter at least one AR coefficient.")
    if text.startswith("[") or text.startswith("("):
        values = ast.literal_eval(text)
    else:
        values = [float(part.strip()) for part in text.split(",")]
    phi = np.atleast_1d(np.asarray(values, dtype=float))
    if phi.ndim != 1 or len(phi) == 0 or not np.all(np.isfinite(phi)):
        raise ValueError("AR coefficients must be a finite one-dimensional sequence.")
    return phi

In [ ]:
def run_experiment(phi, n, intercept, innovation_half_width, burnin, seed,
                   max_points=2500):
    stationary, eigvals = is_stationary(phi)
    if not stationary:
        raise ValueError(
            "The selected AR coefficients are not stationary. "
            f"Companion eigenvalue magnitudes: {np.round(np.abs(eigvals), 4)}"
        )
    x = simulate_uniform_ar(
        phi=phi, sample_size=n, intercept=intercept,
        half_width=innovation_half_width, burnin=burnin, rng=seed,
    )
    cloud2 = sliding_window_point_cloud(x, 2, newest_first=False)
    cloud3 = sliding_window_point_cloud(x, 3, newest_first=False)
    cloud2_plot = cloud2
    cloud3_plot = cloud3
    if len(cloud2) > max_points:
        idx2 = np.linspace(0, len(cloud2) - 1, max_points, dtype=int)
        cloud2_plot = cloud2[idx2]
    if len(cloud3) > max_points:
        idx3 = np.linspace(0, len(cloud3) - 1, max_points, dtype=int)
        cloud3_plot = cloud3[idx3]

    print(f"AR order p = {len(phi)}")
    print(f"Companion eigenvalues = {np.round(eigvals, 4)}")
    print(f"Sample mean = {x.mean():.4f}; sample SD = {x.std(ddof=1):.4f}")
    print(
        "Uniform innovation support = "
        f"[{-innovation_half_width:.4f}, {innovation_half_width:.4f}]"
    )
    if len(x) > max_points:
        print(f"Plots are thinned for responsiveness; full arrays retain all {len(x):,} observations.")

    plot_time_series(x, phi)
    plot_embedding_k2(cloud2_plot, phi, intercept, innovation_half_width)
    plot_embedding_k3(cloud3_plot, phi, intercept, innovation_half_width)
    return x, cloud2, cloud3

## Interactive controls

Enter coefficients in lag order, for example `0.7` for AR(1) or `0.6, -0.2` for AR(2). The innovations are uniformly distributed on $[-a,a]$, where $a$ is the selected innovation half-width. The sample-size field accepts values up to 1,000,000. For long series, plots are automatically thinned for responsiveness, while the complete arrays remain available as `x`, `cloud_k2`, and `cloud_k3`.

In [ ]:
phi_widget = widgets.Text(value="0.7", description="AR coefficients:",
                          style={"description_width": "130px"}, layout=widgets.Layout(width="360px"))
n_widget = widgets.BoundedIntText(value=1200, min=200, max=1_000_000, step=1000,
                                 description="Sample size:",
                                 style={"description_width": "130px"},
                                 layout=widgets.Layout(width="300px"))
half_width_widget = widgets.FloatSlider(value=1.0, min=0.1, max=3.0, step=0.1,
                                        description="Innovation half-width:",
                                        style={"description_width": "150px"})
intercept_widget = widgets.FloatSlider(value=0.0, min=-2.0, max=2.0, step=0.1,
                                       description="Intercept:", style={"description_width": "130px"})
seed_widget = widgets.IntText(value=7, description="Random seed:",
                              style={"description_width": "130px"})
run_button = widgets.Button(description="Generate and plot", button_style="primary", icon="play")
output = widgets.Output()


def handle_run(_=None):
    with output:
        clear_output(wait=True)
        try:
            phi = parse_phi(phi_widget.value)
            global x, cloud_k2, cloud_k3
            x, cloud_k2, cloud_k3 = run_experiment(
                phi=phi,
                n=n_widget.value,
                intercept=intercept_widget.value,
                innovation_half_width=half_width_widget.value,
                burnin=max(500, 10 * len(phi)),
                seed=seed_widget.value,
            )
        except Exception as exc:
            print(f"Could not generate the requested process: {exc}")


run_button.on_click(handle_run)
controls = widgets.VBox([
    widgets.HTML("<b>AR process settings</b>"),
    phi_widget,
    widgets.HBox([n_widget, half_width_widget]),
    widgets.HBox([intercept_widget, seed_widget]),
    run_button,
])
display(controls, output)
handle_run()

## Non-widget alternative

If widgets are unavailable in your Jupyter frontend, edit and run the cell below directly.

In [ ]:
# Example: an AR(2) with uniform innovations.
# Uncomment to run.
# phi = np.array([0.6, -0.2])
# x, cloud_k2, cloud_k3 = run_experiment(
#     phi=phi,
#     n=1500,
#     intercept=0.0,
#     innovation_half_width=1.0,
#     burnin=500,
#     seed=7,
# )